In [ ]:
# ==============================================================================
# MỤC 1: SỬ DỤNG THƯ VIỆN (SCIKIT-LEARN) CHO THUẬT TOÁN K-MEANS CLUSTERING
# ==============================================================================

# Import thư viện numpy để tính toán mảng số học và đại số tuyến tính
import numpy as np
# Import lớp KMeans từ thư viện scikit-learn cho bài toán phân cụm không giám sát
from sklearn.cluster import KMeans
# Import các độ đo đánh giá chất lượng phân cụm (Silhouette Score và Adjusted Rand Index)
from sklearn.metrics import silhouette_score, adjusted_rand_score

def run_kmeans_sklearn(X, n_clusters=3, max_iter=300, random_state=42):
    # Khởi tạo mô hình KMeans với số cụm k (n_clusters), số vòng lặp tối đa và hạt giống ngẫu nhiên
    # init='k-means++' giúp chọn tâm cụm ban đầu thông minh để hội tụ nhanh và ổn định hơn
    model = KMeans(
        n_clusters=n_clusters,
        init='k-means++',
        n_init=10,
        max_iter=max_iter,
        random_state=random_state
    )
    
    # Huấn luyện mô hình và gán nhãn cụm trực tiếp cho từng điểm dữ liệu trong X
    labels = model.fit_predict(X)
    
    # Lấy tọa độ các tâm cụm (centroids) sau khi mô hình hội tụ
    centroids = model.cluster_centers_
    
    # Lấy giá trị Inertia (tổng bình phương khoảng cách từ các điểm đến tâm cụm gần nhất - WCSS)
    inertia = model.inertia_
    
    # Trả về mô hình, nhãn cụm dự đoán, tọa độ tâm cụm và giá trị inertia
    return model, labels, centroids, inertia


In [ ]:
# ==============================================================================
# MỤC 2: TỰ VIẾT CODE THUẬT TOÁN K-MEANS TỪ ĐẦU (FROM SCRATCH)
# ==============================================================================

class CustomKMeans:
    def __init__(self, n_clusters=3, max_iter=300, tol=1e-4, random_state=42):
        # Khởi tạo các siêu tham số cho thuật toán K-Means:
        # n_clusters: số lượng cụm (k) cần phân chia
        # max_iter: số vòng lặp tối đa cho phép
        # tol: ngưỡng sai số hội tụ (nếu tâm cụm dịch chuyển nhỏ hơn tol thì dừng sớm)
        # random_state: hạt giống ngẫu nhiên để đảm bảo tính tái lập kết quả
        self.n_clusters = n_clusters
        self.max_iter = max_iter
        self.tol = tol
        self.random_state = random_state
        self.centroids = None
        self.labels_ = None
        self.inertia_ = None

    def _initialize_centroids(self, X):
        # Đặt hạt giống ngẫu nhiên
        np.random.seed(self.random_state)
        n_samples = X.shape[0]
        # Chọn ngẫu nhiên k chỉ số điểm dữ liệu không trùng lặp từ tập X làm tâm cụm ban đầu
        random_indices = np.random.choice(n_samples, size=self.n_clusters, replace=False)
        return X[random_indices].copy()

    def _compute_distances(self, X, centroids):
        # Tính khoảng cách Euclid từ mỗi điểm dữ liệu trong X tới từng tâm cụm
        # X có kích thước (n_samples, n_features), centroids có kích thước (n_clusters, n_features)
        # Sử dụng broadcasting để tính ma trận khoảng cách kích thước (n_samples, n_clusters)
        distances = np.linalg.norm(X[:, np.newaxis, :] - centroids[np.newaxis, :, :], axis=2)
        return distances

    def _assign_clusters(self, distances):
        # Gán mỗi điểm dữ liệu vào cụm có tâm gần nhất (chỉ số của khoảng cách nhỏ nhất)
        return np.argmin(distances, axis=1)

    def _update_centroids(self, X, labels):
        # Khởi tạo mảng mới lưu tọa độ các tâm cụm vừa cập nhật
        new_centroids = np.zeros((self.n_clusters, X.shape[1]), dtype=np.float64)
        for k in range(self.n_clusters):
            # Lọc tất cả các điểm dữ liệu hiện đang thuộc về cụm k
            cluster_points = X[labels == k]
            if len(cluster_points) > 0:
                # Tâm cụm mới là trung bình cộng tọa độ của tất cả các điểm trong cụm k
                new_centroids[k] = cluster_points.mean(axis=0)
            else:
                # Xử lý trường hợp cụm bị rỗng: giữ nguyên tâm cụm cũ
                new_centroids[k] = self.centroids[k]
        return new_centroids

    def _compute_inertia(self, X, labels, centroids):
        # Tính tổng bình phương khoảng cách (WCSS / Inertia) từ mỗi điểm đến tâm cụm của nó
        inertia = 0.0
        for k in range(self.n_clusters):
            cluster_points = X[labels == k]
            if len(cluster_points) > 0:
                inertia += np.sum((cluster_points - centroids[k]) ** 2)
        return inertia

    def fit(self, X):
        # Bước 1: Khởi tạo tọa độ các tâm cụm ban đầu
        self.centroids = self._initialize_centroids(X)

        # Vòng lặp tối ưu hóa luân phiên giữa 2 bước: Gán cụm (E-step) và Cập nhật tâm (M-step)
        for iteration in range(self.max_iter):
            # Bước 2: Tính khoảng cách từ các điểm tới tâm cụm hiện tại và gán nhãn cụm
            distances = self._compute_distances(X, self.centroids)
            labels = self._assign_clusters(distances)

            # Bước 3: Cập nhật lại vị trí tâm cụm mới dựa trên trung bình các điểm trong cụm
            new_centroids = self._update_centroids(X, labels)

            # Bước 4: Kiểm tra điều kiện hội tụ (mức độ dịch chuyển của tâm cụm)
            centroid_shift = np.linalg.norm(new_centroids - self.centroids)
            self.centroids = new_centroids
            self.labels_ = labels

            # Nếu tâm cụm gần như không đổi (nhỏ hơn ngưỡng tol), dừng vòng lặp sớm
            if centroid_shift < self.tol:
                break

        # Tính giá trị Inertia cuối cùng sau khi mô hình hội tụ
        self.inertia_ = self._compute_inertia(X, self.labels_, self.centroids)
        return self

    def predict(self, X):
        # Dự đoán nhãn cụm cho dữ liệu mới dựa trên các tâm cụm đã học được
        distances = self._compute_distances(X, self.centroids)
        return self._assign_clusters(distances)

    def fit_predict(self, X):
        # Huấn luyện mô hình và trả về nhãn cụm trực tiếp
        self.fit(X)
        return self.labels_


In [ ]:
# ==============================================================================
# MỤC 3: TẠO DATA MOCK VÀ KIỂM THỬ SO SÁNH 2 THUẬT TOÁN (THƯ VIỆN VS TỰ VIẾT)
# ==============================================================================

# Import hàm make_blobs để tạo bộ dữ liệu giả lập (mock data) có cấu trúc phân cụm rõ ràng
from sklearn.datasets import make_blobs

# Bước 1: Tạo dữ liệu giả lập (Mock Data)
# Tạo 500 điểm dữ liệu trong không gian 2 chiều (2 đặc trưng), chia thành 4 cụm thực tế
X, y_true = make_blobs(
    n_samples=500,
    n_features=2,
    centers=4,
    cluster_std=1.1,
    random_state=42
)

print(f"Kích thước tập dữ liệu Mock: {X.shape}, Số cụm thực tế: {len(np.unique(y_true))}\n")

# Bước 2: Chạy thử nghiệm phân cụm với thư viện Scikit-Learn (Mục 1)
sklearn_model, labels_sklearn, centroids_sklearn, inertia_sklearn = run_kmeans_sklearn(
    X, n_clusters=4, random_state=42
)

# Bước 3: Chạy thử nghiệm phân cụm với thuật toán tự viết CustomKMeans (Mục 2)
custom_model = CustomKMeans(n_clusters=4, max_iter=300, random_state=42)
labels_custom = custom_model.fit_predict(X)
centroids_custom = custom_model.centroids
inertia_custom = custom_model.inertia_

# Bước 4: Đánh giá và so sánh hiệu năng giữa 2 phương pháp
# Lưu ý: Chỉ số nhãn cụm (0, 1, 2, 3) có thể hoán vị giữa 2 lần chạy nên ta dùng ARI và Silhouette Score để so sánh
ari_between_models = adjusted_rand_score(labels_sklearn, labels_custom)
ari_sklearn_vs_true = adjusted_rand_score(y_true, labels_sklearn)
ari_custom_vs_true = adjusted_rand_score(y_true, labels_custom)

sil_sklearn = silhouette_score(X, labels_sklearn)
sil_custom = silhouette_score(X, labels_custom)

print("=== KẾT QUẢ ĐÁNH GIÁ & SO SÁNH K-MEANS ===")
print(f"Inertia (WCSS) - Thư viện Sklearn  : {inertia_sklearn:.4f}")
print(f"Inertia (WCSS) - Thuật toán tự viết : {inertia_custom:.4f}\n")

print(f"Silhouette Score - Thư viện Sklearn : {sil_sklearn:.4f}")
print(f"Silhouette Score - Thuật toán tự viết: {sil_custom:.4f}\n")

print(f"Adjusted Rand Index (so với nhãn gốc) - Sklearn : {ari_sklearn_vs_true:.4f}")
print(f"Adjusted Rand Index (so với nhãn gốc) - Tự viết : {ari_custom_vs_true:.4f}")
print(f"Độ tương đồng phân cụm giữa 2 mô hình (ARI)     : {ari_between_models * 100:.2f}%\n")

# Sắp xếp tâm cụm theo tọa độ X để dễ dàng đối chiếu trực tiếp giữa 2 thuật toán
sorted_centroids_sklearn = centroids_sklearn[np.argsort(centroids_sklearn[:, 0])]
sorted_centroids_custom = centroids_custom[np.argsort(centroids_custom[:, 0])]

print("Tọa độ 4 tâm cụm (đã sắp xếp) - Thư viện Sklearn:")
print(np.round(sorted_centroids_sklearn, 4))
print("\nTọa độ 4 tâm cụm (đã sắp xếp) - Thuật toán tự viết:")
print(np.round(sorted_centroids_custom, 4))

# Kiểm tra khả năng dự đoán trên điểm dữ liệu mới chưa từng thấy
new_samples = np.array([[-8.0, 7.0], [4.5, 2.0], [-2.5, 9.0]])
print("\nDự đoán cụm cho 3 điểm dữ liệu mới:")
print("Sklearn dự đoán :", sklearn_model.predict(new_samples))
print("Tự viết dự đoán :", custom_model.predict(new_samples))
